<!-- nav -->
[← 6 · Saving and loading: snapshots](06_Saving_And_Loading.ipynb) · [Home](../../README.md) · [8 · Tools that work for every schema →](08_Tools_For_Every_Schema.ipynb)

# 7 · JSON for machines, YAML for people

**The problem.** An office directory is served to a web app as JSON and maintained by the operations team as a
YAML file in git. Two classic bugs are waiting:

- **The Norway problem.** In YAML 1.1, which most YAML libraries still implement, the unquoted country code `NO`
  reads as the boolean `false`. So do `no`, `off` and `n`, and `yes`/`on`/`y` read as `true`.
- **Leading zeros.** Oslo's postal code `0150` reads as the number 150, or as octal. `1:30` reads as ninety
  (sexagesimal). `2026-09-01` becomes a date object.

JSON has its own traps: `NaN` and `Infinity` are not JSON, but Python's `json` writes them anyway. And integers above
2^53 silently lose precision in JavaScript.

Since case study 6 all data passes through plain data with exact types, so the text layer only has to write and read
that faithfully. `JSON` and `YAML` mirror `Plain` exactly: `ToJSON(store)(schema, value)`, `FromJSON(store)(schema, text)`,
with `.Reachable` for graphs.

In [1]:
import math

import yaml  # PyYAML, only to show what a stock YAML 1.1 reader does

from mbse.Schemas.Framework import JSON, YAML, Proxies, Schemas, Validators
from mbse.Schemas.Framework.Errors import DecodeError

store = Proxies.OfStore()

Office = (
    Schemas.OfObject.Builder().name('Office').ref()
    .properties(
        lambda prop: prop.name('city').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('country_code').of(lambda t: t.as_native(str)),  # ISO 3166-1 alpha-2
        lambda prop: prop.name('postal_code').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('opens').of(lambda t: t.as_native(str)),          # local time, 'HH:MM'
        lambda prop: prop.name('desks').of(lambda t: t.as_native(int)),
        lambda prop: prop.name('occupancy').of(lambda t: t.as_native(float)),
        lambda prop: prop.name('open_plan').of(lambda t: t.as_native(bool)),
    )
    .create()
)
Staff = Schemas.OfObject.Builder().name('Staff').ref().properties(lambda prop: prop.name('name').of(lambda t: t.as_native(str))).create()
Seating = (
    Schemas.OfRelation.Builder().name('Seating')
    .links('office', 'person')
    .properties(lambda prop: prop.name('desk').of(lambda t: t.as_native(str)))
    .unique('office')  # each person sits in one office
    .create()
)
Office = Schemas.OfObject.Builder(Office).relations(lambda adj: adj.name('staff').of(Seating).me('office')).update()
Staff = Schemas.OfObject.Builder(Staff).relations(lambda adj: adj.name('office').of(Seating).me('person')).update()
for schema in [Office, Staff, Seating]:
    store.register(schema)

oslo = (
    store.Office()
    .city('Oslo').country_code('NO').postal_code('0150').opens('1:30')
    .desks(2**60).occupancy(math.nan).open_plan(False)
    .staff(lambda x: x.person(lambda p: p.name('Ingrid')).desk('A-01'))
    .create()
)

## JSON

`JSON.ToJSON` writes a snapshot as JSON text. `indent` is optional:

In [2]:
text = JSON.ToJSON(store).Reachable(Office, oslo, indent=2)
print(text)

{
  "root": "s0",
  "objects": {
    "s0": {
      "city": "Oslo",
      "country_code": "NO",
      "postal_code": "0150",
      "opens": "1:30",
      "desks": 1152921504606846976,
      "occupancy": "NaN",
      "open_plan": false,
      "staff": [
        {
          "person": {
            "$ref": "s1",
            "$schema": "Staff"
          },
          "desk": "A-01"
        }
      ]
    },
    "s1": {
      "name": "Ingrid",
      "office": [
        {
          "office": {
            "$ref": "s0",
            "$schema": "Office"
          },
          "desk": "A-01"
        }
      ]
    }
  }
}


- `"NO"`, `"0150"` and `"1:30"` are strings because the schema says so. JSON quotes strings anyway.
- `NaN` became the *string* `"NaN"`, so the output is strict JSON that any parser accepts. The float schema knows to
  read it back as NaN.
- `2**60` is written exactly. Whether a JavaScript reader keeps it is the reader's business. The TypeScript
  implementation of this framework reads it as a `bigint`.
- Key order follows the schema and the snapshot, and non-ASCII text is written as is.

Reading is strict in the same spirit. Text that other parsers accept with a shrug is refused here, because
accepting it would mean guessing:

In [3]:
load = JSON.FromJSON(store)
for why, bad in [
    ('NaN literal', '{"root": "s0", "objects": {"s0": {"occupancy": NaN}}}'),
    ('duplicate key', '{"root": "s0", "root": "s1", "objects": {}}'),
    ('int for a float', '{"root": "s0", "objects": {"s0": {"occupancy": 1}}}'),
    ('truncated', '{"root": "s0", "objects": '),
]:
    try:
        load(Office, bad)
    except DecodeError as error:
        print(f'{why:16} -> {error}')

NaN literal      -> line 1, column 48: NaN is not valid JSON
duplicate key    -> line 1, column 16: duplicate key 'root'
int for a float  -> $.objects.s0.occupancy: expected float, got int
truncated        -> line 1, column 27: expecting a value


Every problem is a `DecodeError` (case study 6), located where it was found. Problems in the *text* (a `NaN`
literal, a duplicate key, a truncated document) have a line and column. The wrong type is only found once the
text has become plain data and meets the schema, so it has a path. Columns count characters, not bytes or UTF-16
units, so an editor's "go to line and column" lands on the spot.

**Why so strict about `1` for a float?** Because another language may not have one number type. If `1` were quietly
accepted as `1.0`, the same document could mean different things in Python and in a language where integers and
floats differ. Duplicate keys are refused for the same reason: parsers disagree about which one wins.

The output is also **byte-identical** across implementations. The TypeScript binding writes exactly the same text
for the same data, which is checked by a shared conformance corpus. You can diff, hash or sign a document without
caring which language produced it.

In [4]:
back = load.Reachable(Office, text)
print(back.country_code, back.postal_code, back.opens, back.desks == 2**60, math.isnan(back.occupancy))

NO 0150 1:30 True True


## YAML

`YAML.ToYAML` writes the same snapshot as YAML:

In [5]:
print(YAML.ToYAML(store).Reachable(Office, oslo))

root: s0
objects:
  s0:
    city: Oslo
    country_code: 'NO'
    postal_code: '0150'
    opens: '1:30'
    desks: 1152921504606846976
    occupancy: NaN
    open_plan: false
    staff:
    - person:
        $ref: s1
        $schema: Staff
      desk: A-01
  s1:
    name: Ingrid
    office:
    - office:
        $ref: s0
        $schema: Office
      desk: A-01



Every string a YAML reader might misread is quoted: `'NO'`, `'0150'`, `'1:30'`, `'NaN'`. The rule is "quote if a
YAML 1.1 **or** 1.2 reader could take it for anything but this string", so the file is safe whichever library the
next reader uses. A stock YAML 1.1 loader, PyYAML's `safe_load`, reads it back correctly:

In [6]:
stock = yaml.safe_load(YAML.ToYAML(store).Reachable(Office, oslo))
stock['objects']['s0']['country_code'], stock['objects']['s0']['postal_code'], stock['objects']['s0']['opens']

('NO', '0150', '1:30')

## Hand-written YAML

People don't quote carefully. Here is the ops team's file, typed by hand:

In [7]:
handwritten = """
root: s0
objects:
  s0:
    city: Oslo
    country_code: NO        # YAML 1.1: false. YAML 1.2: the string "NO"
    postal_code: 0150       # YAML 1.1: 104 (octal). YAML 1.2 core: 150, an int
    opens: 1:30             # YAML 1.1: 90. YAML 1.2: the string "1:30"
    desks: 40
    open_plan: false
"""

print('stock YAML 1.1 reader:', {k: v for k, v in yaml.safe_load(handwritten)['objects']['s0'].items()})

stock YAML 1.1 reader: {'city': 'Oslo', 'country_code': False, 'postal_code': 104, 'opens': 90, 'desks': 40, 'open_plan': False}


The framework loads YAML with the **YAML 1.2 core schema**. Only `true`/`false` are booleans, and `1:30` and dates are
strings. That fixes `NO` and `1:30`. `0150` is still an integer in YAML 1.2, and the schema says `postal_code` is a
string. Since types are never coerced, loading refuses rather than turning 150 into `'150'` and silently losing the
zero:

In [8]:
from_yaml = YAML.FromYAML(store)
try:
    from_yaml(Office, handwritten)
except DecodeError as error:
    print('DecodeError:', error)

DecodeError: $.objects.s0.postal_code: expected str, got int


Quote it, and it loads:

In [9]:
office = from_yaml(Office, handwritten.replace('0150 ', "'0150'"))
print(office.country_code, office.postal_code, office.opens, office.open_plan)
print(Validators.Validate(store)(Office, office))

NO 0150 1:30 False
[]


That's the pattern for configuration files: load with the framework, and a wrong type is an error with a location,
never a quiet reinterpretation. YAML features that don't map to plain data are refused too: tags like `!!binary`,
several documents in one file, and non-string keys. Anchors and aliases are accepted on input and never written.

In [10]:
for why, bad in [('tag', 'root: s0\nobjects:\n  s0:\n    city: !!binary AAAA\n'),
                 ('two documents', '---\nroot: s0\n---\nroot: s1\n'),
                 ('non-string key', '1: x\n')]:
    try:
        from_yaml(Office, bad)
    except DecodeError as error:
        print(f'{why:15} -> {error}')

tag             -> line 4, column 11: unsupported tag !!binary
two documents   -> line 3, column 1: expected a single document, found another
non-string key  -> line 1, column 1: keys must be strings, got int 1


These are the framework's own YAML rules, so their messages are the same in every language binding. Plain syntax
errors (an unclosed bracket, a tab used for indentation) are reported as a `DecodeError` with a line and column too,
but the wording comes from each binding's YAML parser:

In [11]:
try:
    from_yaml(Office, 'root: [s0\n')
except DecodeError as error:
    print(error)

line 2, column 1: while parsing a flow sequence, expected ',' or ']', but got '<stream end>'


## Plain data in, plain data out

Both modules also expose `loads`/`dumps` for plain data without a schema. They're handy for inspecting files, or
for converting between the two formats:

In [12]:
plain = YAML.loads(YAML.ToYAML(store).Reachable(Office, oslo))
print(JSON.dumps(plain) == JSON.ToJSON(store).Reachable(Office, oslo))

True


## What you learned

- `JSON` and `YAML` mirror `Plain`: `ToJSON`/`ToYAML(store)(schema, value)`, `FromJSON`/`FromYAML(store)(schema, text)`,
  and `.Reachable`.
- JSON output is strict (non-finite floats become strings), exact for big integers, and byte-identical across
  language bindings.
- YAML is written so both 1.1 and 1.2 readers agree, and read with the 1.2 core schema. The Norway problem is gone.
- Types are never coerced on input. A value of the wrong type is an error, not a guess.
- Every input problem is a `DecodeError`: a line and column for text, or a path for plain data.

**Next:** so far you've used tools the framework provides. Case study 8 writes a new one, a generic "diff two
snapshots" and "describe any object" utility, and shows why the visitor protocol makes that possible without
knowing any schema in advance.

<!-- nav -->
[← 6 · Saving and loading: snapshots](06_Saving_And_Loading.ipynb) · [Home](../../README.md) · [8 · Tools that work for every schema →](08_Tools_For_Every_Schema.ipynb)